# Problem Set 3
## INF266- Reinforcement Learning


## Part 1

# 1 Monte Carlo methods

#### Task 1.8:

Both estimates change downwards when redefining x_2 as x_1 + uniformly drawn noise.
In the first case, the samples are drawn uniformly over the square, so the fraction of points inside the circle correctly estimates the area ratio 𝜋/4. The estimator is therefore unbiased, and the only error comes from Monte Carlo variance.
In the second case, the samples are no longer uniformly distributed because 𝑥2	​depends on x1. The points are concentrated along a diagonal band instead of covering the square evenly. As a result, the computed fraction inside the circle no longer represents the true area ratio, and the estimator becomes biased.

# 2 MC and MABs

#### Task 2.1
A multi-armed bandit is a control problem.
The objective is not only to estimate expected rewards, but to choose actions in order to maximize cumulative reward. This corresponds to policy optimization.
#### Task 2.2:
The expected rewards of arms correspond to an action-value function.
In a MAB there is effectively only one state, so: Q(a)=E[R∣a]
We estimate the expected reward of each arm.
#### Task 2.3:
Normally a trajectory is: (State,Action,Reward,State)
In a MAB, there are no state transitions, so a trajectory reduces to (Action,reward)
A collection of trajectories is simply a sequence of independent action–reward pairs.
#### Task 2.4:
MC control.
Estimates Q(a) via sample averages,
Improve the policy by acting greedily with respect to Q
This matches the MC control framework (policy evaluation + policy improvement).
#### Task 2.5:
MC prediction corresponds to estimating: Q(a)=E[R∣a]
under a fixed policy, without performing policy improvement.
#### Task 2.6:
In both MABs and Monte Carlo control, ε ensures exploration.
In MABs, ε prevents premature commitment to a suboptimal arm.
In Monte Carlo control, ε-soft policies guarantee sufficient exploration of all state–action pairs, which is required for convergence.
#### Task 2.7:
Yes, a standard reinforcement learning problem can be reduced to a MAB by removing state transitions and environment dynamics.
However, this removes:

    Sequential decision structure
    State-dependent values
    Long-term credit assignment
    Planning over time
The problem becomes a single-state, stateless decision problem.

# 3 Importance Sampling

#### Task 3.4:


In [35]:
import numpy as np
rewards_R = [0, 5, 10, 20] #$
probs_R   = [.35, .3, .25, .1]

rewards_S = [0, 5, 10, 20] #$
probs_S   = [.3, .35, .3, .05]

rho = np.array(probs_R) / np.array(probs_S)
print("\nRatio between the distribution probability of R and S:")
for reward, r in zip(rewards_R, rho):
    print(f"Reward {reward}: rho = {r}")



Ratio between the distribution probability of R and S:
Reward 0: rho = 1.1666666666666667
Reward 5: rho = 0.8571428571428572
Reward 10: rho = 0.8333333333333334
Reward 20: rho = 2.0


#### Task 3.5:

In [36]:
#helpers from 3.1 and 3.3 in order to run 3.5
expected_R = np.sum(np.array(rewards_R) * np.array(probs_R))
samples_S = np.random.choice(rewards_S, size=10000, p=probs_S)

#3.5
ratio_dict = dict(zip(rewards_R, rho))
weights = np.array([ratio_dict[x] for x in samples_S])

IS_estimate_R = np.mean(weights * samples_S)

print("\nIS estimate of E[R] using samples from S:")
print(f"E_hat_IS[R]: {IS_estimate_R}")
print(f"True E[R]: {expected_R}")
print(f"IS Error: {IS_estimate_R - expected_R}")


IS estimate of E[R] using samples from S:
E_hat_IS[R]: 5.874738095238095
True E[R]: 6.0
IS Error: -0.12526190476190457


#### Task 3.8: # CHECK!!!--------------------------
From caomparison of E[R]: 
From S:   5.983595238095238
From S (R_prime)':  5.9786
From S (R_double)': 6.613326653306612


The results differ because the variance of importance sampling depends on how close the behavior distribution is to the target distribution.
When sampling from S, which is similar to R, the importance weights remain moderate and the estimate of 
E[R] is stable. With S′, all rewards have reasonable probability, so the estimator remains fairly stable.
However, with S″, some rewards have very small probability under the behavior distribution but higher probability under R. This produces large importance weights and therefore high variance, causing the estimate to fluctuate significantly.
In general, the larger the mismatch between behavior and target distribution, the higher the variance of the importance sampling estimator.

# 4 Q-Learning


#### Task 4.1

We do not need importance sampling in Q-learning because the update does not attempt to evaluate the expected return of a target policy using trajectories generated by a different behavior policy.

Instead, Q-learning applies the Bellman optimality backup:
Q(s,a)← Q(s,a)+α[r+γ max Q(s′,a′)− Q(s,a)]

The target depends on the maximum estimated action-value in the next state, not on the action actually taken by the behavior policy. Therefore, the update is independent of the behavior policy’s action probabilities.
Q-learning is inherently off-policy, it learns the optimal value function. Since it bootstraps toward the optimal value rather than estimating expectations under a specific target policy, importance sampling ratios are not required.

## Part 2

In [37]:
#Code from 5.1-5.3 needed to run 5.4
from collections import defaultdict
from pathlib import Path
import numpy as np
import pickle

traj_path = Path("/Users/endrefangel/Desktop/INF266/trajectories.pickle") # Bytt med riktig PATH!!!!

with open(traj_path, "rb") as f:
    trajectories = pickle.load(f)

gamma = 1.0  # No discounting, total accumulated reward

def MC_action_value(trajectories, gamma=1.0):
    return_sum = defaultdict(float)
    return_count = defaultdict(int)

    for episode in trajectories:
        G = 0.0
        returns_from_t = [0.0] * len(episode)

        for t in reversed(range(len(episode))):
            _, _, r_t1, _ = episode[t]
            G = r_t1 + gamma * G
            returns_from_t[t] = G

        visited = set()
        for t, (s, a, _, _) in enumerate(episode):
            sa = (s, a)
            if sa not in visited:
                visited.add(sa)
                return_sum[sa] += returns_from_t[t]
                return_count[sa] += 1

    return {sa: return_sum[sa] / return_count[sa] for sa in return_sum}

q_pi = MC_action_value(trajectories, gamma)

#### Task 5.4
Yes, we can perform MC improvement by defining a new policy
pi'(s) = arg max_a q_pi(s,a)
that is, choosing the action with the highest estimated action-value for each state.

However, the new policy is not guaranteed to be optimal. The estimates were computed from offline trajectories generated by the original policy. After improvement, the new policy may select actions that were rarely or never taken in the dataset, meaning their value estimates may be inaccurate. Without collecting new trajectories and repeating evaluation and improvement, optimality cannot be guaranteed.


In [38]:
#whole clue with mc-improvement, as we remember from lecture is that we can select an action that maximizes q_pi(s,a) for each state s to get an improved policy.
def MC_improvement_from_q(q_pi):
    q_by_state = defaultdict(dict)
    for (state, action), q_val in q_pi.items():
        q_by_state[state][action] = q_val

    improved_policy = {}
    for state, action_values in q_by_state.items():
        best_action = max(action_values, key=action_values.get)
        improved_policy[state] = best_action
    return improved_policy, q_by_state

improved_policy, q_by_state = MC_improvement_from_q(q_pi)
print("States with improved action:", len(improved_policy))

States with improved action: 2924


#### Task 5.5
Not all trajectories are equally useful for Monte Carlo evaluation and improvement. The quality of the estimates depends on how often each state–action pair is visited: frequently visited pairs yield low-variance estimates, while rarely visited pairs remain noisy or undefined. Trajectories that explore diverse states and actions contribute more information than repetitive ones following the same path. Longer trajectories are only more valuable if they cover new state–action pairs. Since the dataset was generated under a fixed policy, poorly explored actions cannot be reliably compared, which limits the quality of the improvement step.

#### Task 5.6
At this point in the hw, we have no simulator for M, only trajectories, which are not sufficient for implementing MC-control

#### Task 5.7
We choose an ε-greedy policy with relatively high exploration. The environment is stochastic and contains a large state space (31×100 grid), making sufficient exploration necessary for reliable estimation of action-values. Since Monte Carlo control updates only after full episodes and relies entirely on observed returns, inadequate exploration would result in poor state–action coverage and unreliable estimates. A sufficiently exploratory starting policy therefore reduces the risk of premature convergence to suboptimal paths and improves the stability of early learning.

#### Task 5.9
MC control improved performance significantly and learned near-optimal behavior. However, convergence was very slow and computationally expensive due to high variance, long episodes, and stochastic transitions. The policy approached optimality but was not perfectly converged. The main challenges were sample inefficiency and computational cost.

#### 5.12
We evaluated the algorithms using episode return and a moving average over a fixed window (200 episodes) to smooth variability and capture the learning trend. From the moving-average learning curve, SARSA(0) learns faster and produces a more stable improvement pattern than Monte Carlo. MC has higher variance because it relies on full-episode returns, while SARSA(0) updates online using bootstrapping, which improves learning stability and sample efficiency. Overall, SARSA(0) outperformed MC in learning speed and stability.

# CODE FOR 5.12????

In [ ]:
import numpy as np
from collections import defaultdict

N_ACTIONS = 5
FORWARD_ACTIONS = [0, 2, 4]
OTHER_ACTIONS = [1, 3]

def starting_policy(state):
    probs = np.zeros(N_ACTIONS, dtype=float)
    probs[FORWARD_ACTIONS] = 0.9 / len(FORWARD_ACTIONS)
    probs[OTHER_ACTIONS] = 0.1 / len(OTHER_ACTIONS)
    return probs

def generate_episode(env, policy):
    obs, _ = env.reset()
    state = tuple(obs["agent"]["pos"])
    episode = []
    total_reward = 0.0

    while True:
        probs = policy(state)
        action = np.random.choice(N_ACTIONS, p=probs)
        obs, reward, terminated, truncated, _ = env.step(action)
        episode.append((state, action, reward))
        total_reward += reward
        state = tuple(obs["agent"]["pos"])
        if terminated or truncated:
            break

    return episode, total_reward

def mc_control(env, n_episodes=50_000, gamma=1.0, epsilon=0.1, tol=1e-4, patience=200):
    Q = defaultdict(lambda: np.zeros(N_ACTIONS, dtype=float))
    N = defaultdict(lambda: np.zeros(N_ACTIONS, dtype=float))
    stable = 0
    ep_returns = np.empty(n_episodes, dtype=float)

    eps_base = np.full(N_ACTIONS, epsilon / N_ACTIONS, dtype=float)

    def policy(s, _Q=Q):
        probs = eps_base.copy()
        probs[int(np.argmax(_Q[s]))] += 1.0 - epsilon
        return probs

    for ep in range(n_episodes):
        episode, total_reward = generate_episode(env, policy)
        ep_returns[ep] = total_reward

        G = 0.0
        visited = set()
        max_delta = 0.0

        for state, action, reward in reversed(episode):
            G = gamma * G + reward
            sa = (state, action)
            if sa in visited:
                continue
            visited.add(sa)

            N_s = N[state]
            Q_s = Q[state]
            N_s[action] += 1.0
            old = Q_s[action]
            Q_s[action] += (G - old) / N_s[action]
            delta = abs(Q_s[action] - old)
            if delta > max_delta:
                max_delta = delta

        stable = stable + 1 if max_delta < tol else 0
        if stable >= patience:
            print(f"Converged at episode {ep + 1} (max_delta={max_delta:.2e})")
            ep_returns = ep_returns[: ep + 1]
            break

    def greedy_policy(s, _Q=Q):
        probs = np.zeros(N_ACTIONS, dtype=float)
        probs[int(np.argmax(_Q[s]))] = 1.0
        return probs

    return Q, greedy_policy, ep_returns

def train_sarsa_with_history(env, n_episodes=10_000, gamma=1.0, alpha=0.1, epsilon=0.1):
    n_actions = env.action_space.n
    Q_hist = defaultdict(lambda: np.zeros(n_actions, dtype=float))
    episode_returns = np.empty(n_episodes, dtype=float)

    def eps_greedy_action_local(state):
        if np.random.rand() < epsilon:
            return np.random.randint(n_actions)
        return int(np.argmax(Q_hist[state]))

    for ep in range(n_episodes):
        obs, _ = env.reset()
        state = tuple(obs["agent"]["pos"])
        action = eps_greedy_action_local(state)
        total_reward = 0.0

        while True:
            obs, reward, terminated, truncated, _ = env.step(action)
            next_state = tuple(obs["agent"]["pos"])
            done = terminated or truncated
            total_reward += reward

            if done:
                td_target = reward
            else:
                next_action = eps_greedy_action_local(next_state)
                td_target = reward + gamma * Q_hist[next_state][next_action]

            Q_hist[state][action] += alpha * (td_target - Q_hist[state][action])

            if done:
                break

            state, action = next_state, next_action

        episode_returns[ep] = total_reward

    return Q_hist, episode_returns

In [ ]:
#load envs
import gymnasium

env_v1 = gymnasium.make('mountain/GridWorld-v1')
env_2 = gymnasium.make('mountain/GridWorld-v2')

In [ ]:
import matplotlib.pyplot as plt

# train SARSA(0) once; reuse if already computed
if "sarsa_returns" not in globals():
    Q_sarsa_plot, sarsa_returns = train_sarsa_with_history(
        env_2, n_episodes=30_000, alpha=0.1, epsilon=0.1
    )

window = 200
kernel = np.ones(window) / window
sarsa_ma = np.convolve(sarsa_returns, kernel, mode="valid")

plt.figure(figsize=(12, 5))
plt.plot(sarsa_returns, alpha=0.25, label="Episode return")
plt.plot(np.arange(window - 1, len(sarsa_returns)), sarsa_ma, linewidth=2, label=f"Moving average ({window})")
plt.title("SARSA(0) on mountain/GridWorld-v2")
plt.xlabel("Episode")
plt.ylabel("Return")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
# train MC once; reuse if already computed
if "mc_returns" not in globals():
    Q_v2, optimal_policy_v2, mc_returns = mc_control(env_2, n_episodes=30_000)

window = 200
mc_ma = np.convolve(mc_returns, np.ones(window) / window, mode="valid")

plt.figure(figsize=(12, 5))
plt.plot(mc_returns, alpha=0.25, label="Episode return")
plt.plot(np.arange(window - 1, len(mc_returns)), mc_ma, linewidth=2, label=f"Moving average ({window})")
plt.title("MC Control on mountain/GridWorld-v2")
plt.xlabel("Episode")
plt.ylabel("Return")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


#### Task 5.14
#### Task 5.16